In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, models

In [5]:
import glob
files = glob.glob("*.csv")

df_list = []

for file in files:
    temp = pd.read_csv(file, low_memory=False)
    temp.columns = temp.columns.str.strip()

    # keep only files that have Label column
    if "Label" in temp.columns:
        df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Initial shape:", df.shape)
print(df["Label"].value_counts().head(20))

Initial shape: (1992804, 80)
Label
Benign           1544621
Bot               286191
Infilteration     161934
Label                 58
Name: count, dtype: int64


In [6]:
target_col = "Label"

# Clean column names
df.columns = df.columns.str.strip()

# Remove rows where Label is missing
df = df[df[target_col].notna()]

# Convert label to string and strip spaces
df[target_col] = df[target_col].astype(str).str.strip()

# Remove accidental header rows ONLY if present
df = df[df[target_col].str.lower() != "label"]

print("After label cleaning:", df.shape)
print(df[target_col].value_counts())

After label cleaning: (1992746, 80)
Label
Benign           1544621
Bot               286191
Infilteration     161934
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["Encoded_Label"] = label_encoder.fit_transform(df[target_col])

print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'Benign': 0, 'Bot': 1, 'Infilteration': 2}


In [8]:
# Remove timestamp columns
for col in df.columns:
    if "time" in col.lower() or "timestamp" in col.lower():
        df.drop(columns=[col], inplace=True)

# Remove duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# Remove known duplicate column
if "Fwd Header Length.1" in df.columns:
    df.drop(columns=["Fwd Header Length.1"], inplace=True)

# Replace infinity
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Convert features to numeric
for col in df.columns:
    if col != target_col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with missing numeric values
df.dropna(inplace=True)

# Drop duplicate rows
df.drop_duplicates(inplace=True)

print("Final shape:", df.shape)
print(df[target_col].value_counts())

Final shape: (1583592, 80)
Label
Benign           1299282
Bot               144535
Infilteration     139775
Name: count, dtype: int64


In [9]:
print("Dataset Shape:", df.shape)

print("\nLabel Distribution:")
print(df[target_col].value_counts())

print("\nNull Values:")
print(df.isnull().sum().sum())

print("\nInfinity Values:")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (1583592, 80)

Label Distribution:
Label
Benign           1299282
Bot               144535
Infilteration     139775
Name: count, dtype: int64

Null Values:
0

Infinity Values:
6110

Duplicate Rows:
0


In [10]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [11]:
print("\nInfinity Values:")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())


Infinity Values:
0


In [12]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["Encoded_Label"] = label_encoder.fit_transform(df[target_col])

print("\nClass Encoding:")
print(dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))


Class Encoding:
{'Benign': 0, 'Bot': 1, 'Infilteration': 2}


In [13]:
target_col = "Encoded_Label"

# Drop original string Label and keep encoded target
X = df.drop(columns=["Label", "Encoded_Label"])
y = df["Encoded_Label"]

# Convert all features to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Clean again
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)

# Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "MI_Score": mi_scores
}).sort_values(by="MI_Score", ascending=False)

top_k = int(len(X.columns) * 0.25)
selected_features = mi_df.head(top_k)["Feature"].tolist()

print("Selected Features:")
print(selected_features)

Selected Features:
['Pkt Len Max', 'Bwd Pkt Len Max', 'Fwd Pkt Len Max', 'Dst Port', 'Pkt Len Std', 'Pkt Len Mean', 'Bwd Pkt Len Std', 'Fwd Pkt Len Std', 'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Bwd Seg Size Avg', 'Pkt Len Var', 'Init Bwd Win Byts', 'Pkt Size Avg', 'TotLen Bwd Pkts', 'Subflow Bwd Byts', 'Subflow Fwd Byts', 'TotLen Fwd Pkts']


In [14]:
target_col = "Encoded_Label"

df_selected = X[selected_features].copy()
df_selected[target_col] = y

print(df_selected.shape)
print(df_selected[target_col].value_counts())

(1583592, 20)
Encoded_Label
0    1299282
1     144535
2     139775
Name: count, dtype: int64


In [15]:
X_gan = df_selected.drop(columns=[target_col])
y_gan = df_selected[target_col].astype(int)

In [16]:
from sklearn.preprocessing import StandardScaler

scaler_gan = StandardScaler()

X_gan_scaled = scaler_gan.fit_transform(X_gan)

In [17]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

X_tensor = torch.tensor(X_gan_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_gan.values, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True
)

In [18]:
input_dim = X_tensor.shape[1]

num_classes = len(np.unique(y_gan))

latent_dim = 64

print("Input Features:", input_dim)
print("Classes:", num_classes)

Input Features: 19
Classes: 3


In [19]:
class Generator(nn.Module):
    def __init__(self, latent_dim, num_classes, output_dim):
        super().__init__()

        self.label_emb = nn.Embedding(num_classes, num_classes)

        self.model = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 128),
            nn.LeakyReLU(0.2),

            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),

            nn.Linear(512, output_dim)
        )

    def forward(self, noise, labels):
        label_input = self.label_emb(labels)
        x = torch.cat([noise, label_input], dim=1)
        return self.model(x)

In [20]:
class Discriminator(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.label_emb = nn.Embedding(num_classes, num_classes)

        self.model = nn.Sequential(
            nn.Linear(input_dim + num_classes, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, data, labels):
        label_input = self.label_emb(labels)
        x = torch.cat([data, label_input], dim=1)
        return self.model(x)

In [21]:
import torch.optim as optim
generator = Generator(latent_dim, num_classes, input_dim)
discriminator = Discriminator(input_dim, num_classes)

criterion = nn.BCELoss()

g_optimizer = optim.Adam(generator.parameters(), lr=0.0002)
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002)


c:\Users\Aditya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
epochs = 10

d_losses = []
g_losses = []

for epoch in range(epochs):
    epoch_d_loss = 0
    epoch_g_loss = 0
    batches = 0

    for real_data, real_labels in loader:
        batch_size = real_data.size(0)

        valid = torch.ones(batch_size, 1)
        fake = torch.zeros(batch_size, 1)

        # Train Discriminator
        noise = torch.randn(batch_size, latent_dim)
        gen_labels = real_labels
        fake_data = generator(noise, gen_labels)

        real_output = discriminator(real_data, real_labels)
        fake_output = discriminator(fake_data.detach(), gen_labels)

        d_loss_real = criterion(real_output, valid)
        d_loss_fake = criterion(fake_output, fake)
        d_loss = d_loss_real + d_loss_fake

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Train Generator
        noise = torch.randn(batch_size, latent_dim)
        gen_labels = real_labels
        fake_data = generator(noise, gen_labels)

        output = discriminator(fake_data, gen_labels)
        g_loss = criterion(output, valid)

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()
        batches += 1

    avg_d_loss = epoch_d_loss / batches
    avg_g_loss = epoch_g_loss / batches

    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    print(f"Epoch [{epoch+1}/{epochs}] | D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f}")

Epoch [1/10] | D Loss: 1.1244 | G Loss: 1.1426
Epoch [2/10] | D Loss: 1.1800 | G Loss: 0.9466
Epoch [3/10] | D Loss: 1.2060 | G Loss: 0.9013
Epoch [4/10] | D Loss: 1.2082 | G Loss: 0.8919
Epoch [5/10] | D Loss: 1.2037 | G Loss: 0.8964
Epoch [6/10] | D Loss: 1.1938 | G Loss: 0.9184
Epoch [7/10] | D Loss: 1.2125 | G Loss: 0.8778
Epoch [8/10] | D Loss: 1.2213 | G Loss: 0.8640
Epoch [9/10] | D Loss: 1.2276 | G Loss: 0.8579
Epoch [10/10] | D Loss: 1.2519 | G Loss: 0.8360


In [23]:
class_counts = df_selected["Encoded_Label"].value_counts()
max_count = class_counts.max()

print("Before GAN balancing:")
print(class_counts)

Before GAN balancing:
Encoded_Label
0    1299282
1     144535
2     139775
Name: count, dtype: int64


In [24]:
synthetic_parts = [df_selected]

for class_label, count in class_counts.items():
    needed = max_count - count

    print(f"Class {class_label} needs {needed} synthetic samples")

    if needed > 0:
        noise = torch.randn(needed, latent_dim)

        labels = torch.full(
            (needed,),
            int(class_label),
            dtype=torch.long
        )

        synthetic_scaled = generator(noise, labels).detach().numpy()

        synthetic_original = scaler_gan.inverse_transform(synthetic_scaled)

        synthetic_df = pd.DataFrame(
            synthetic_original,
            columns=X_gan.columns
        )

        synthetic_df["Encoded_Label"] = int(class_label)

        synthetic_parts.append(synthetic_df)

balanced_df = pd.concat(
    synthetic_parts,
    axis=0,
    ignore_index=True
)

balanced_df["Encoded_Label"] = balanced_df["Encoded_Label"].astype(int)

print("After GAN balancing:")
print(balanced_df["Encoded_Label"].value_counts())

Class 0 needs 0 synthetic samples
Class 1 needs 1154747 synthetic samples
Class 2 needs 1159507 synthetic samples
After GAN balancing:
Encoded_Label
0    1299282
2    1299282
1    1299282
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier

In [26]:
target_col = "Encoded_Label"

X = balanced_df.drop(columns=[target_col])
y = balanced_df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [27]:
num_classes = len(label_encoder.classes_)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [28]:
y_pred = xgb_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted"))

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9639981015175032
F1 Score: 0.9639004427503721
               precision    recall  f1-score   support

       Benign       0.90      1.00      0.95    259856
          Bot       1.00      1.00      1.00    259857
Infilteration       1.00      0.89      0.94    259857

     accuracy                           0.96    779570
    macro avg       0.97      0.96      0.96    779570
 weighted avg       0.97      0.96      0.96    779570

[[259679      3    174]
 [    67 259790      0]
 [ 27822      0 232035]]


In [29]:
results_df = pd.DataFrame({
    "Actual_Label": y_test.values,
    "Predicted_Label": y_pred
})

results_df["Actual_Attack"] = label_encoder.inverse_transform(results_df["Actual_Label"])
results_df["Predicted_Attack"] = label_encoder.inverse_transform(results_df["Predicted_Label"])

results_df.head(20)

,Actual_Label,Predicted_Label,Actual_Attack,Predicted_Attack
0,2,2,Infilteration,Infilteration
1,0,0,Benign,Benign
2,2,2,Infilteration,Infilteration
3,1,1,Bot,Bot
4,0,0,Benign,Benign
5,2,2,Infilteration,Infilteration
6,1,1,Bot,Bot
7,0,0,Benign,Benign
8,2,2,Infilteration,Infilteration
9,2,2,Infilteration,Infilteration


In [30]:
wrong_preds = results_df[results_df["Actual_Attack"] != results_df["Predicted_Attack"]]

print("Wrong Predictions:", len(wrong_preds))
wrong_preds.head(20)

Wrong Predictions: 28066


,Actual_Label,Predicted_Label,Actual_Attack,Predicted_Attack
67,2,0,Infilteration,Benign
92,2,0,Infilteration,Benign
138,2,0,Infilteration,Benign
185,2,0,Infilteration,Benign
190,2,0,Infilteration,Benign
198,2,0,Infilteration,Benign
227,2,0,Infilteration,Benign
286,2,0,Infilteration,Benign
289,2,0,Infilteration,Benign
307,2,0,Infilteration,Benign


In [31]:
def predict_attack(sample_df):
    sample = sample_df[selected_features]

    sample_scaled = scaler.transform(sample)

    pred = xgb_model.predict(sample_scaled)
    prob = xgb_model.predict_proba(sample_scaled)

    attack_name = label_encoder.inverse_transform(pred)[0]
    confidence = prob.max()

    return {
        "Prediction": attack_name,
        "Confidence": round(float(confidence), 4)
    }

In [32]:
sample = df_selected.drop(columns=["Encoded_Label"]).sample(1, random_state=42)

predict_attack(sample)

{'Prediction': 'Benign', 'Confidence': 0.931}

In [33]:
# Get one real Infilteration sample
infiltration_label = label_encoder.transform(["Infilteration"])[0]

infiltration_sample = df_selected[
    df_selected["Encoded_Label"] == infiltration_label
].drop(columns=["Encoded_Label"]).sample(1, random_state=42)

print(infiltration_sample)

        Pkt Len Max  Bwd Pkt Len Max  Fwd Pkt Len Max  Dst Port  Pkt Len Std  \
551483            0                0                0       443          0.0   

        Pkt Len Mean  Bwd Pkt Len Std  Fwd Pkt Len Std  Fwd Seg Size Avg  \
551483           0.0              0.0              0.0               0.0   

        Fwd Pkt Len Mean  Bwd Pkt Len Mean  Bwd Seg Size Avg  Pkt Len Var  \
551483               0.0               0.0               0.0          0.0   

        Init Bwd Win Byts  Pkt Size Avg  TotLen Bwd Pkts  Subflow Bwd Byts  \
551483                  0           0.0              0.0                 0   

        Subflow Fwd Byts  TotLen Fwd Pkts  
551483                 0                0  


In [34]:
prediction = predict_attack(infiltration_sample)

print(prediction)

{'Prediction': 'Benign', 'Confidence': 0.7819}


In [35]:
print("Actual Attack: Infilteration")
print("Predicted:", prediction)

Actual Attack: Infilteration
Predicted: {'Prediction': 'Benign', 'Confidence': 0.7819}


In [36]:
print(label_encoder.classes_)

['Benign' 'Bot' 'Infilteration']


In [37]:
print(confusion_matrix(y_test, y_pred))

[[259679      3    174]
 [    67 259790      0]
 [ 27822      0 232035]]


In [38]:
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))

               precision    recall  f1-score   support

       Benign       0.90      1.00      0.95    259856
          Bot       1.00      1.00      1.00    259857
Infilteration       1.00      0.89      0.94    259857

     accuracy                           0.96    779570
    macro avg       0.97      0.96      0.96    779570
 weighted avg       0.97      0.96      0.96    779570



In [39]:
def predict_attack_safe(sample_df, threshold=0.85):
    sample = sample_df[selected_features]
    sample_scaled = scaler.transform(sample)

    probs = xgb_model.predict_proba(sample_scaled)[0]
    pred_class = np.argmax(probs)

    attack_name = label_encoder.inverse_transform([pred_class])[0]
    confidence = probs[pred_class]

    if confidence < threshold:
        return {
            "Prediction": "Suspicious / Needs Review",
            "Model_Prediction": attack_name,
            "Confidence": round(float(confidence), 4)
        }

    return {
        "Prediction": attack_name,
        "Confidence": round(float(confidence), 4)
    }

In [43]:
infiltration_label = label_encoder.transform(["Infilteration"])[0]

infiltration_data = df_selected[
    df_selected["Encoded_Label"] == infiltration_label
].drop(columns=["Encoded_Label"])

# Remove weak all-zero samples
strong_infiltration = infiltration_data[
    infiltration_data.sum(axis=1) > 0
]

# Pick a stronger sample
infiltration_test = strong_infiltration.sample(1, random_state=10)

print(infiltration_test)

        Pkt Len Max  Bwd Pkt Len Max  Fwd Pkt Len Max  Dst Port  Pkt Len Std  \
880010            0                0                0        80          0.0   

        Pkt Len Mean  Bwd Pkt Len Std  Fwd Pkt Len Std  Fwd Seg Size Avg  \
880010           0.0              0.0              0.0               0.0   

        Fwd Pkt Len Mean  Bwd Pkt Len Mean  Bwd Seg Size Avg  Pkt Len Var  \
880010               0.0               0.0               0.0          0.0   

        Init Bwd Win Byts  Pkt Size Avg  TotLen Bwd Pkts  Subflow Bwd Byts  \
880010                 -1           0.0              0.0                 0   

        Subflow Fwd Byts  TotLen Fwd Pkts  
880010                 0                0  


In [44]:
result = predict_attack_safe(infiltration_test)

print("Actual Attack: Infilteration")
print("Prediction Result:")
print(result)

Actual Attack: Infilteration
Prediction Result:
{'Prediction': 'Benign', 'Confidence': 0.9573}


In [5]:
import joblib

joblib.dump(xgb_model, "multiclass_xgb_nids.pkl")
joblib.dump(scaler, "multiclass_scaler.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(selected_features, "selected_features.pkl")

NameError: name 'xgb_model' is not defined

In [46]:
def predict_attack_safe(sample_df, threshold=0.85):
    sample = sample_df[selected_features]
    sample_scaled = scaler.transform(sample)

    probs = xgb_model.predict_proba(sample_scaled)[0]
    pred_class = np.argmax(probs)

    attack_name = label_encoder.inverse_transform([pred_class])[0]
    confidence = probs[pred_class]

    if confidence < threshold:
        return {
            "Prediction": "Suspicious / Needs Review",
            "Model_Prediction": attack_name,
            "Confidence": round(float(confidence), 4)
        }

    return {
        "Prediction": attack_name,
        "Confidence": round(float(confidence), 4)
    }

In [47]:
for attack in label_encoder.classes_:
    attack_label = label_encoder.transform([attack])[0]

    sample = df_selected[
        df_selected["Encoded_Label"] == attack_label
    ].drop(columns=["Encoded_Label"]).sample(1, random_state=42)

    print("\nActual:", attack)
    print(predict_attack_safe(sample))


Actual: Benign
{'Prediction': 'Benign', 'Confidence': 0.9303}

Actual: Bot
{'Prediction': 'Bot', 'Confidence': 1.0}

Actual: Infilteration
{'Prediction': 'Suspicious / Needs Review', 'Model_Prediction': 'Benign', 'Confidence': 0.7819}


In [4]:
from xgboost import XGBClassifier

num_classes = len(label_encoder.classes_)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=num_classes,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

NameError: name 'label_encoder' is not defined

In [1]:
import os

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

In [2]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

c:\Users\Aditya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import joblib
import shap

# Load saved XGBoost model
xgb_model = joblib.load(
    r"C:\Users\Aditya\Desktop\VECC Intern\AI NIDS Project\multiclass_xgb_nids.pkl"
)

# Create SHAP explainer
explainer = shap.TreeExplainer(xgb_model)

ValueError: could not convert string to float: '[5.9604645E-7,-3.5762787E-7,-3.5762787E-7]'

Trails on SHAP


In [10]:
X_test_df = pd.DataFrame(
    X_test_scaled,
    columns=selected_features
)

NameError: name 'X_test_scaled' is not defined

In [8]:
import xgboost
import pandas as pd
import numpy as np

X_test_df = pd.DataFrame(
    X_test_scaled,
    columns=selected_features
)

dmatrix = xgboost.DMatrix(
    X_test_df,
    feature_names=selected_features
)

shap_values = xgb_model.get_booster().predict(
    dmatrix,
    pred_contribs=True
)

print(shap_values.shape)

NameError: name 'X_test_scaled' is not defined